# Sesión 02 - Probabilidad condicional y probabilidad total

Objetivo: calcular probabilidades condicionales, reconstruir probabilidades con particiones y actualizar creencias con Bayes.


In [ ]:
from itertools import product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.precision", 4)


## 1. Probabilidad condicional en dos dados


### Lectura matemática

- **Distribución asumida:** medida uniforme discreta sobre $\Omega=\{1,\ldots,6\}^2$.
- **Parámetro estimado:** ninguno; se calcula $P(A\mid B)$ por conteo exacto y normalización dentro de $B$.
- **Supuesto que puede fallar:** la condición $B$ debe tener probabilidad positiva y representar información realmente observada.
- **Diagnóstico:** verificar $P(A\cap B)=P(A\mid B)P(B)$ y comparar $P(A\mid B)$ contra $P(A)$ para discutir dependencia.


In [ ]:
omega = np.array(list(product(range(1, 7), repeat=2)))
p = np.full(len(omega), 1 / len(omega))
suma = omega.sum(axis=1)

def prob(evento: np.ndarray) -> float:
    return float(p[evento].sum())

def cond_prob(evento: np.ndarray, condicion: np.ndarray) -> float:
    if prob(condicion) == 0:
        raise ValueError("La condición tiene probabilidad cero.")
    return prob(evento & condicion) / prob(condicion)

A = suma >= 9
B = omega[:, 0] >= 4
C = suma % 2 == 0

pd.DataFrame(
    {
        "cantidad": ["P(A)", "P(A|B)", "P(A|C)", "P(B|A)"],
        "valor": [prob(A), cond_prob(A, B), cond_prob(A, C), cond_prob(B, A)],
    }
)


## 2. Teorema de probabilidad total

Usamos como partición el valor del primer dado.


### Lectura matemática

- **Distribución asumida:** partición finita $B_1,\ldots,B_k$ con probabilidades que suman 1.
- **Parámetro estimado:** no se estima; se reconstruye $P(A)$ mediante $\sum_i P(A\mid B_i)P(B_i)$.
- **Supuesto que puede fallar:** los bloques deben ser mutuamente excluyentes y exhaustivos; si falta un bloque, el total queda sesgado.
- **Diagnóstico:** comprobar $\sum_i P(B_i)=1$ y que el cálculo directo de $P(A)$ coincida con la suma ponderada.


In [ ]:
particion = [omega[:, 0] == cara for cara in range(1, 7)]
componentes = []

for cara, bloque in enumerate(particion, start=1):
    componentes.append(
        {
            "B_i": f"dado_1={cara}",
            "P(B_i)": prob(bloque),
            "P(A|B_i)": cond_prob(A, bloque),
            "producto": cond_prob(A, bloque) * prob(bloque),
        }
    )

total = pd.DataFrame(componentes)
print("P(A) directo:", round(prob(A), 4))
print("P(A) por probabilidad total:", round(total["producto"].sum(), 4))
total


## 3. Caso diagnóstico

Una prueba médica no se interpreta solo con sensibilidad y especificidad. La prevalencia cambia fuertemente el posterior.


### Lectura matemática

- **Distribución asumida:** variable Bernoulli para enfermedad y Bernoulli para resultado del test.
- **Parámetros usados:** prevalencia, sensibilidad y especificidad.
- **Supuesto que puede fallar:** sensibilidad/especificidad no constantes por subpoblación.
- **Diagnóstico:** análisis de sensibilidad del posterior al cambiar prevalencia y tasas de error.


In [ ]:
prevalencia = 0.02
sensibilidad = 0.95          # P(+|D)
especificidad = 0.90         # P(-|no D)

p_d = prevalencia
p_no_d = 1 - p_d
p_pos_d = sensibilidad
p_pos_no_d = 1 - especificidad

p_pos = p_pos_d * p_d + p_pos_no_d * p_no_d
p_d_pos = p_pos_d * p_d / p_pos

print(f"P(+) = {p_pos:.4f}")
print(f"P(D|+) = {p_d_pos:.4f}")


In [ ]:
prevalencias = np.linspace(0.001, 0.30, 100)
posteriores = []
for prev in prevalencias:
    p_pos = sensibilidad * prev + (1 - especificidad) * (1 - prev)
    posteriores.append(sensibilidad * prev / p_pos)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(prevalencias, posteriores)
ax.set_title("Efecto de la prevalencia sobre P(D|+)")
ax.set_xlabel("prevalencia P(D)")
ax.set_ylabel("posterior P(D|+)")
ax.grid(alpha=0.3)
plt.show()


## 4. Actualización bayesiana secuencial

La presentación enfatiza que la evidencia puede llegar en más de una señal. Si las señales son condicionalmente independientes dado el estado real, cada evidencia multiplica los odds por su razón de verosimilitudes. Si no son independientes, hay que modelar $P(E_2\mid H,E_1)$ explícitamente.


### Lectura matemática

- **Distribución asumida:** dos resultados positivos de test modelados como Bernoulli condicionalmente independientes dado enfermedad/no enfermedad.
- **Parámetros usados:** prevalencia, sensibilidad, especificidad y razón de verosimilitudes positiva $LR_+=P(+\mid D)/P(+\mid D^c)$.
- **Supuesto que puede fallar:** dos tests pueden compartir tecnología, muestra o sesgo; en ese caso no son evidencias independientes.
- **Diagnóstico:** comparar actualización secuencial contra cálculo conjunto y revisar si el posterior crece de forma plausible.


In [ ]:
lr_positivo = sensibilidad / (1 - especificidad)
odds_prior = p_d / p_no_d

odds_1 = odds_prior * lr_positivo
p_d_pos_odds = odds_1 / (1 + odds_1)

p_d_dos_pos_directo = (sensibilidad**2 * p_d) / (sensibilidad**2 * p_d + (1 - especificidad)**2 * p_no_d)

p_d_dos_pos_secuencial = (sensibilidad * p_d_pos) / (
    sensibilidad * p_d_pos + (1 - especificidad) * (1 - p_d_pos)
)

odds_2 = odds_prior * lr_positivo**2
p_d_dos_pos_odds = odds_2 / (1 + odds_2)

pd.DataFrame(
    {
        "cantidad": [
            "prior P(D)",
            "LR+",
            "P(D|+) por Bayes",
            "P(D|+) por odds",
            "P(D|+,+) directo",
            "P(D|+,+) secuencial",
            "P(D|+,+) por odds",
        ],
        "valor": [
            p_d,
            lr_positivo,
            p_d_pos,
            p_d_pos_odds,
            p_d_dos_pos_directo,
            p_d_dos_pos_secuencial,
            p_d_dos_pos_odds,
        ],
    }
)


## 5. Valor esperado de información: perfecta e imperfecta

Este bloque conecta la probabilidad condicional con decisiones bajo incertidumbre. La información perfecta observa el estado real; la información imperfecta observa una señal ruidosa y requiere actualizar posteriores con Bayes.


In [ ]:
estados = pd.DataFrame(
    {
        "estado": ["demanda_baja", "demanda_media", "demanda_alta"],
        "probabilidad": [0.25, 0.50, 0.25],
    }
)

pagos = pd.DataFrame(
    {
        "accion": ["no_lanzar", "lanzar_pequeño", "lanzar_grande"],
        "demanda_baja": [0, -10, -40],
        "demanda_media": [0, 35, 45],
        "demanda_alta": [0, 55, 120],
    }
)

probs = estados.set_index("estado")["probabilidad"]
valor_esperado = pagos.set_index("accion").mul(probs, axis=1).sum(axis=1)
mejor_sin_info = valor_esperado.max()
valor_con_info_perfecta = pagos.set_index("accion").max(axis=0).mul(probs).sum()
evpi = valor_con_info_perfecta - mejor_sin_info

print(valor_esperado)
print(f"Mejor valor sin información: {mejor_sin_info:.2f}")
print(f"Valor con información perfecta: {valor_con_info_perfecta:.2f}")
print(f"EVPI: {evpi:.2f}")


### Información imperfecta: valor esperado de una señal

Con una señal $S$ no se conoce directamente la demanda; se actualiza $P(\theta\mid S=s)$ y se decide de forma óptima para cada señal. El valor esperado de la señal debe ser menor o igual al EVPI.


### Lectura matemática

- **Distribución asumida:** estados de demanda categóricos y señal categórica con matriz $P(S=s\mid\theta)$.
- **Parámetros usados:** priors de demanda, matriz de verosimilitud de la señal y matriz de pagos.
- **Supuesto que puede fallar:** la señal puede estar mal calibrada o cambiar en el tiempo; entonces sus likelihoods no son estables.
- **Diagnóstico:** verificar que cada columna de $P(S\mid\theta)$ sume 1, que $0\le EVSI\le EVPI$ y que el valor neto reste el costo de adquirir la señal.


In [ ]:
pagos_index = pagos.set_index("accion")

likelihood_senal = pd.DataFrame(
    {
        "demanda_baja": [0.65, 0.25, 0.10],
        "demanda_media": [0.20, 0.55, 0.25],
        "demanda_alta": [0.10, 0.25, 0.65],
    },
    index=["desfavorable", "neutral", "favorable"],
)

assert np.allclose(likelihood_senal.sum(axis=0), 1.0)

conjunta_senal_estado = likelihood_senal.mul(probs, axis=1)
p_senal = conjunta_senal_estado.sum(axis=1)
posterior_estado_dado_senal = conjunta_senal_estado.div(p_senal, axis=0)

filas = []
for senal, posterior in posterior_estado_dado_senal.iterrows():
    valores = pagos_index.mul(posterior, axis=1).sum(axis=1)
    mejor_accion = valores.idxmax()
    filas.append(
        {
            "senal": senal,
            "P(senal)": p_senal.loc[senal],
            "mejor_accion": mejor_accion,
            "valor_posterior": valores.loc[mejor_accion],
        }
    )

politica_senal = pd.DataFrame(filas)
valor_con_senal = (politica_senal["P(senal)"] * politica_senal["valor_posterior"]).sum()
evsi = valor_con_senal - mejor_sin_info
costo_senal = 8.0

print("Posteriores P(estado | señal):")
print(posterior_estado_dado_senal.round(3).to_string())
print(f"Valor con señal imperfecta: {valor_con_senal:.2f}")
print(f"EVSI: {evsi:.2f}")
print(f"EVPI: {evpi:.2f}")
print(f"Valor neto si la señal cuesta {costo_senal:.2f}: {evsi - costo_senal:.2f}")
assert 0 <= evsi <= evpi + 1e-12
politica_senal


## 6. Naive Bayes como probabilidad condicional aplicada

El material anterior llevaba probabilidad condicional hacia Bayes y Naive Bayes. Aquí se muestra una implementación mínima desde tablas de frecuencia. La idea clave es factorizar la verosimilitud y trabajar en log-probabilidades para evitar underflow:

$$
P(Y\mid X_1,\ldots,X_p) \propto P(Y)\prod_j P(X_j\mid Y)
$$


Los scores se normalizan con una operación equivalente a softmax sobre log-scores, de modo que las probabilidades posteriores por clase sumen 1.


In [ ]:
entrenamiento = pd.DataFrame(
    {
        "clima": ["sol", "sol", "nublado", "lluvia", "lluvia", "lluvia", "nublado", "sol", "sol", "lluvia"],
        "promo": ["si", "no", "si", "si", "no", "no", "no", "si", "no", "si"],
        "compra": [1, 1, 1, 1, 0, 0, 1, 1, 0, 0],
    }
)

def naive_bayes_predict(tabla, observacion, target="compra", alpha=1.0):
    clases = sorted(tabla[target].unique())
    features = [c for c in tabla.columns if c != target]
    scores = {}
    for clase in clases:
        subset = tabla[tabla[target] == clase]
        log_score = np.log(len(subset) / len(tabla))
        for feature in features:
            niveles = tabla[feature].nunique()
            cuenta = ((subset[feature] == observacion[feature]).sum())
            prob_feature = (cuenta + alpha) / (len(subset) + alpha * niveles)
            log_score += np.log(prob_feature)
        scores[clase] = log_score
    max_score = max(scores.values())
    probs = {k: np.exp(v - max_score) for k, v in scores.items()}
    total = sum(probs.values())
    return {k: v / total for k, v in probs.items()}

observacion = {"clima": "lluvia", "promo": "si"}
naive_bayes_predict(entrenamiento, observacion)


## 7. Titanic: Naive Bayes categórico desde cero

Este bloque recupera el ejercicio legacy de Titanic. La variable objetivo es `survived` y los predictores se discretizan para usar un Naive Bayes categórico con suavizado de Laplace.

La regla de decisión es:

$$
\hat{y}=\arg\max_y \left[\log P(Y=y)+\sum_j \log P(X_j=x_j\mid Y=y)\right]
$$


### Lectura matemática

- **Distribución asumida:** $Y$ es Bernoulli y cada predictor discretizado sigue una distribución categórica condicional a $Y$.
- **Parámetros estimados:** $P(Y)$ y tablas $P(X_j=x\mid Y=y)$ con suavizado de Laplace.
- **Supuesto que puede fallar:** independencia condicional entre predictores.
- **Diagnóstico:** comparar log-loss/accuracy contra `CategoricalNB` y revisar variables con fuga de información.


In [ ]:
from pathlib import Path

def display(obj):
    try:
        from IPython.display import display as ipy_display
        ipy_display(obj)
    except Exception:
        if hasattr(obj, "to_string"):
            print(obj.to_string())
        else:
            print(obj)

def encontrar_data_dir():
    for candidato in [Path("data_sources"), Path("../data_sources")]:
        if candidato.exists():
            return candidato
    return None

DATA_DIR = encontrar_data_dir()
if DATA_DIR is None or not (DATA_DIR / "titanic.csv").exists():
    print("No se encontró data_sources/titanic.csv. El bloque Titanic se omite.")
else:
    titanic = pd.read_csv(DATA_DIR / "titanic.csv")
    display(titanic[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "alone"]].head())


In [ ]:
def preparar_titanic_categorico(df: pd.DataFrame) -> pd.DataFrame:
    data = df[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "alone"]].copy()
    data["age"] = data["age"].fillna(data["age"].median())
    data["fare"] = data["fare"].fillna(data["fare"].median())
    data["embarked"] = data["embarked"].fillna(data["embarked"].mode()[0])
    data["age_bin"] = pd.cut(data["age"], bins=[0, 12, 18, 35, 60, 100], labels=False, include_lowest=True)
    data["fare_bin"] = pd.qcut(data["fare"], q=5, labels=False, duplicates="drop")
    data["family_size"] = data["sibsp"] + data["parch"] + 1
    data["family_bin"] = pd.cut(data["family_size"], bins=[0, 1, 3, 20], labels=False, include_lowest=True)
    data = data[["survived", "pclass", "sex", "age_bin", "fare_bin", "embarked", "alone", "family_bin"]]
    for col in data.columns:
        data[col] = data[col].astype("category").cat.codes
    return data

class NaiveBayesCategorico:
    def __init__(self, alpha: float = 1.0):
        self.alpha = alpha

    def fit(self, X: np.ndarray, y: np.ndarray, n_categories: list[int]):
        self.classes_, class_counts = np.unique(y, return_counts=True)
        self.class_log_prior_ = np.log(class_counts / class_counts.sum())
        self.feature_log_prob_ = []
        for j, n_cat in enumerate(n_categories):
            counts = np.zeros((len(self.classes_), n_cat), dtype=float)
            for class_idx, cls in enumerate(self.classes_):
                values = X[y == cls, j].astype(int)
                counts[class_idx] = np.bincount(values, minlength=n_cat)
            smoothed = counts + self.alpha
            smoothed = smoothed / smoothed.sum(axis=1, keepdims=True)
            self.feature_log_prob_.append(np.log(smoothed))
        return self

    def predict_log_proba(self, X: np.ndarray) -> np.ndarray:
        log_probs = np.tile(self.class_log_prior_, (X.shape[0], 1))
        for j, feature_log_prob in enumerate(self.feature_log_prob_):
            log_probs += feature_log_prob[:, X[:, j].astype(int)].T
        normalizer = np.logaddexp.reduce(log_probs, axis=1, keepdims=True)
        return log_probs - normalizer

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.classes_[np.argmax(self.predict_log_proba(X), axis=1)]

if DATA_DIR is not None and (DATA_DIR / "titanic.csv").exists():
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, log_loss
    from sklearn.naive_bayes import CategoricalNB

    data_cat = preparar_titanic_categorico(titanic)
    X = data_cat.drop(columns="survived").to_numpy(dtype=int)
    y = data_cat["survived"].to_numpy(dtype=int)
    n_categories = [int(X[:, j].max()) + 1 for j in range(X.shape[1])]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    nb_custom = NaiveBayesCategorico(alpha=1.0).fit(X_train, y_train, n_categories)
    pred_custom = nb_custom.predict(X_test)
    proba_custom = np.exp(nb_custom.predict_log_proba(X_test))

    nb_sklearn = CategoricalNB(alpha=1.0, min_categories=n_categories)
    nb_sklearn.fit(X_train, y_train)
    pred_sklearn = nb_sklearn.predict(X_test)
    proba_sklearn = nb_sklearn.predict_proba(X_test)

    resultados_nb_cat = pd.DataFrame(
        {
            "modelo": ["Naive Bayes categórico desde cero", "sklearn CategoricalNB"],
            "accuracy": [accuracy_score(y_test, pred_custom), accuracy_score(y_test, pred_sklearn)],
            "log_loss": [log_loss(y_test, proba_custom), log_loss(y_test, proba_sklearn)],
            "predicciones_iguales_vs_sklearn": [np.array_equal(pred_custom, pred_sklearn), True],
        }
    )
    display(resultados_nb_cat)


## 8. Titanic: Naive Bayes Gaussiano desde cero

Para variables numéricas continuas, Naive Bayes usa una verosimilitud normal por clase y por feature:

$$
P(x_j\mid y=c)=\frac{1}{\sqrt{2\pi\sigma_{jc}^2}}\exp\left(-\frac{(x_j-\mu_{jc})^2}{2\sigma_{jc}^2}\right)
$$


### Lectura matemática

- **Distribución asumida:** $X_j\mid Y=c\sim N(\mu_{jc},\sigma_{jc}^2)$.
- **Parámetros estimados:** medias y varianzas por clase y feature.
- **Supuesto que puede fallar:** normalidad condicional, colas pesadas o variables discretas tratadas como continuas.
- **Diagnóstico:** histogramas por clase, log-loss y comparación contra `GaussianNB`.


In [ ]:
class NaiveBayesGaussiano:
    def __init__(self, var_smoothing: float = 1e-9):
        self.var_smoothing = var_smoothing

    def fit(self, X: np.ndarray, y: np.ndarray):
        self.classes_, counts = np.unique(y, return_counts=True)
        self.class_log_prior_ = np.log(counts / counts.sum())
        self.theta_ = np.vstack([X[y == cls].mean(axis=0) for cls in self.classes_])
        self.var_ = np.vstack([X[y == cls].var(axis=0) for cls in self.classes_])
        self.epsilon_ = self.var_smoothing * np.var(X, axis=0).max()
        self.var_ = self.var_ + self.epsilon_
        return self

    def predict_log_proba(self, X: np.ndarray) -> np.ndarray:
        log_probs = []
        for idx, _cls in enumerate(self.classes_):
            mean = self.theta_[idx]
            var = self.var_[idx]
            log_likelihood = -0.5 * np.sum(np.log(2 * np.pi * var) + ((X - mean) ** 2) / var, axis=1)
            log_probs.append(self.class_log_prior_[idx] + log_likelihood)
        log_probs = np.vstack(log_probs).T
        normalizer = np.logaddexp.reduce(log_probs, axis=1, keepdims=True)
        return log_probs - normalizer

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.classes_[np.argmax(self.predict_log_proba(X), axis=1)]

if DATA_DIR is not None and (DATA_DIR / "titanic.csv").exists():
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, log_loss
    from sklearn.naive_bayes import GaussianNB

    data_num = titanic[["survived", "pclass", "age", "sibsp", "parch", "fare", "alone"]].copy()
    data_num["age"] = data_num["age"].fillna(data_num["age"].median())
    data_num["fare"] = data_num["fare"].fillna(data_num["fare"].median())
    data_num["alone"] = data_num["alone"].astype(int)

    X = data_num.drop(columns="survived").to_numpy(dtype=float)
    y = data_num["survived"].to_numpy(dtype=int)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

    nb_gauss_custom = NaiveBayesGaussiano().fit(X_train, y_train)
    pred_custom = nb_gauss_custom.predict(X_test)
    proba_custom = np.exp(nb_gauss_custom.predict_log_proba(X_test))

    nb_gauss_sklearn = GaussianNB(var_smoothing=1e-9)
    nb_gauss_sklearn.fit(X_train, y_train)
    pred_sklearn = nb_gauss_sklearn.predict(X_test)
    proba_sklearn = nb_gauss_sklearn.predict_proba(X_test)

    resultados_nb_gauss = pd.DataFrame(
        {
            "modelo": ["Naive Bayes gaussiano desde cero", "sklearn GaussianNB"],
            "accuracy": [accuracy_score(y_test, pred_custom), accuracy_score(y_test, pred_sklearn)],
            "log_loss": [log_loss(y_test, proba_custom), log_loss(y_test, proba_sklearn)],
            "predicciones_iguales_vs_sklearn": [np.array_equal(pred_custom, pred_sklearn), True],
        }
    )
    display(resultados_nb_gauss)


## Práctica

1. Cambia prevalencia, sensibilidad y especificidad. Explica qué parámetro afecta más al posterior.
2. Cambia la matriz $P(S\mid\theta)$ de la señal de mercado y verifica si el EVSI sigue siendo menor que el EVPI.
3. En Titanic, modifica las variables usadas por Naive Bayes y revisa si mejora log-loss sin introducir fuga de información.
